# Hide & Seek: a completed study with partial results

I built a 2D hide-and-seek environment, PPO, self-play, and recurrent policies to explore
object use. Some saved agents place and lock boxes near a doorway or gain visibility near
a ramp. I did not establish the intended sequence of construction, purposeful ramp transport,
wall traversal, and effective ramp defense.

This walkthrough uses existing checkpoints. None of its cells starts training.
The [study report](docs/study.md) contains the protocol and limitations.
The [journal](NOTES.md) preserves the development history, including superseded interpretations.


## Selected episodes

Blue agents hide, red agents seek, orange objects are boxes, and the green wedge is a ramp.
The red line marks a sightline that depends on elevation. Ramp proximity changes visibility;
climbing is disabled in these evaluations. These episodes illustrate behaviors and are not
representative success rates.

### Doorway box: tight-room, seed 10
![Selected doorway-box episode](media/barricade.gif)

### Ramp-assisted sightline: room, seed 136
![Selected ramp-sightline episode](media/sightline.gif)

### Ramp lock: two-hiders, seed 25
![Selected ramp-lock episode](media/ramp-lock.gif)

The ramp-lock example is one of six lock-positive episodes in 200. A lock does not establish
successful defense. Matching JSON files beside the GIFs record their seeds and measurements.


## Recorded evaluation

| Configuration | Doorway box (% episodes) | Ramp sightline (% play steps) | Hider ramp lock (% episodes) |
|---|---:|---:|---:|
| `room` | 0.0% | 12.7% | 0.0% |
| `tight-room` | 14.5% | 8.2% | 0.0% |
| `two-hiders` | 26.5% | 10.6% | 3.0% |

- **Doorway box:** percentage of episodes ending with a hider-locked box center inside the
  environment's designated doorway region. This is a placement proxy. It does not verify
  an impassable doorway, a safe shelter, or a complete construction strategy.
- **Ramp sightline:** mean fraction of play steps where at least one active seeker sees a
  hider with elevation and cannot see that same hider when elevation is temporarily removed
  from the visibility query. Another seeker may already see a hider, so this is not necessarily
  an improvement in the team's reward. The query does not advance physics or rerun the episode.
- **Hider ramp lock:** percentage of episodes ending with the hider team owning the ramp lock.
  It does not establish that the lock denied access or improved survival.
- **Hidden fraction:** mean fraction of play steps where every hider is out of sight of all
  active seekers. There is no contact capture in this version of the environment.
- **Ramp displacement:** distance from spawn to final position, plus the greatest such distance
  at any point in the episode. Neither metric identifies which agent moved it or why.

The table mixes episode frequencies and time fractions because the behaviors require different
measurements. It does not report three stages of one successful sequence.



In [ ]:
# Inspect recorded results without rerunning the simulations.
import json
from pathlib import Path

root = Path.cwd()
assert (root / "evaluate.py").exists(), "Open the notebook from the repository root."
for preset in ("room", "tight-room", "two-hiders"):
    report = json.loads((root / "results" / f"{preset}.json").read_text())
    print(preset, report["protocol"]["episodes_per_condition"], "episodes per condition")
    print(json.dumps(report["conditions"]["trained"]["mean"], indent=2))

## Correcting the interpretation

**Proximity was too broad a tool-use measure.** Earlier evaluation counted steps near the ramp.
The current evaluator also asks whether elevation changes a seeker-to-hider sightline. Tests
cover an occluding wall, a clear sightline, a second hider, and an inactive seeker.

**Visibility is not traversal.** In the evaluated task, ramp proximity changes the line-of-sight
mask within a range. It does not change collision rules. Earlier descriptions of seekers
climbing over walls were incorrect for these checkpoints' evaluated mechanics.

**Movement is not purposeful transport.** The new two-hider evaluation measures mean final ramp
displacement of 44.62 pixels and mean maximum displacement of 46.14 pixels in a 600-pixel arena.
The zero-action-hider condition has even larger mean final displacement (58.25 pixels). The
older journal's blanket claim that the ramp never moves does not describe this evaluation.
The measurements do not establish transport toward a useful destination or its strategic value.

**Rare locking is not proof of coordinated defense.** Six of 200 two-hider episodes end with a
hider ramp lock. In seed 14, both the doorway-box and ramp-lock flags are true, yet the hiders
are visible for every play step. Nonzero event counts do not establish successful cooperation.

**The experiments do not prove a general law about emergence.** The journal's claim that
necessity, rather than compute, determines tool use is too strong. Several interventions
changed at once, curricula provided scaffolding, and no multi-seed retraining study isolated
those effects. Failing to obtain a behavior in the attempted runs does not show that scale
cannot help. Likewise, changing team size does not prove a single agent is incapable of
learning a sequence of actions.



## Run a small check

Install dependencies with `uv sync --locked --python 3.13 --extra notebook` and choose
the repository's `.venv` as the kernel. The next cell evaluates two episodes in each condition.
It checks that the pipeline runs; it does not estimate a rare behavior's frequency.


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "evaluate.py", "--preset", "two-hiders", "--episodes", "2"],
    cwd=root,
    check=True,
)

## Reproduce the full results

Run `evaluate.py` with `--episodes 200` for each preset. Save new output files so you can compare
them with the recorded reports. Reports include per-episode rows, runtime versions, source
hashes, and checkpoint hashes. Use `demo.py --preset room --seed 136 --output media/replay.gif`
to reproduce the sightline example.

These are evaluations of one saved policy pair per configuration. They do not measure variance
across independent training runs or isolate the effects of compute, curricula, and geometry.
The project is concluded at this scope. See [README.md](README.md) for setup and
[docs/experiments.md](docs/experiments.md) for the historical file index.
